# 🎬 YT Short Clipper Pro — Colab Edition

**AI-Powered YouTube Shorts Generator** — Jalankan langsung dari Google Colab!

### Fitur:
- 🤖 AI Analysis (Gemini/Groq/OpenRouter)
- 👁️ Face Tracking + Karaoke Subtitle
- 🎨 B-Roll Overlay dari Pexels
- 📐 Split Screen support
- 🎵 Background Music auto-download

### ⚠️ Yang Tidak Tersedia di Colab:
- **Voice Hook** — butuh Voicebox running di komputer lokal

---

**Cara pakai:** Jalankan cell satu per satu dari atas ke bawah.

## 📌 Cell 1: Mount Google Drive

Output video dan config akan disimpan di Google Drive supaya tidak hilang saat runtime disconnect.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create output directory
import os
os.makedirs('/content/drive/MyDrive/YTShortClipper/output', exist_ok=True)
os.makedirs('/content/temp', exist_ok=True)
print('✅ Google Drive mounted! Output folder ready.')

## 📌 Cell 2: Install Dependencies

Install semua library yang dibutuhkan. Cell ini butuh ~2-3 menit.

In [ ]:
# Install system dependencies
!apt-get -qq install ffmpeg

# Install Python dependencies
!pip install -q yt-dlp[default] opencv-python-headless numpy Pillow requests mediapipe python-dotenv
!pip install -q faster-whisper google-genai gradio

print('✅ Semua dependencies terinstall!')

## 📌 Cell 3: Clone Repository & Setup

Clone repo `yt-short-clipper-offline` dan pastikan semua file ada.

In [ ]:
# Clone repository
!git clone https://github.com/Chukie99/yt-short-clipper-offline.git /content/yt-short-clipper-offline 2>/dev/null || true

# Verify files
import os
repo_dir = '/content/yt-short-clipper-offline'
required = ['clipper_core.py', 'clipper_web.py', 'requirements.txt']
for f in required:
    path = os.path.join(repo_dir, f)
    if os.path.exists(path):
        print(f'  ✅ {f}')
    else:
        print(f'  ❌ {f} MISSING!')

# Check fonts directory
fonts_dir = os.path.join(repo_dir, 'fonts')
if os.path.exists(fonts_dir):
    fonts = os.listdir(fonts_dir)
    print(f'  📁 fonts/: {len(fonts)} files')
else:
    print('  ⚠️ fonts/ directory not found — using system fonts')

# Check detector model
detector = os.path.join(repo_dir, 'bin', 'detector.tflite')
if os.path.exists(detector):
    print(f'  ✅ bin/detector.tflite')
else:
    print(f'  ⚠️ bin/detector.tflite not found — face tracking may not work')

# Add repo to Python path
import sys
if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)
print(f'\n✅ Repository ready at {repo_dir}')

## 📌 Cell 4: Setup API Keys (Colab Secrets)

API keys diambil dari **Colab Secrets** (ikon 🔑 di sidebar) supaya aman.

### Cara Setup:
1. Klik ikon 🔑 di sidebar kiri notebook
2. Klik "Add new secret"
3. Tambahkan key: `GEMINI_API_KEY`, `GROQ_API_KEY`, atau `OPENROUTER_API_KEY`
4. Aktifkan "Notebook access" untuk tiap key

Atau, kalau belum setup secrets, cell ini akan meminta input lewat `getpass`.

In [ ]:
import os

# Try Colab Secrets first
try:
    from google.colab import userdata
    
    gemini_key = userdata.get('GEMINI_API_KEY') or ''
    groq_key = userdata.get('GROQ_API_KEY') or ''
    openrouter_key = userdata.get('OPENROUTER_API_KEY') or ''
    
    if gemini_key:
        os.environ['GEMINI_API_KEY'] = gemini_key
        print('✅ GEMINI_API_KEY loaded from Colab Secrets')
    if groq_key:
        os.environ['GROQ_API_KEY'] = groq_key
        print('✅ GROQ_API_KEY loaded from Colab Secrets')
    if openrouter_key:
        os.environ['OPENROUTER_API_KEY'] = openrouter_key
        print('✅ OPENROUTER_API_KEY loaded from Colab Secrets')
        
    if not any([gemini_key, groq_key, openrouter_key]):
        print('⚠️ No secrets found. Using getpass fallback...')
        raise Exception('No secrets')
        
except Exception:
    # Fallback: getpass
    import getpass
    print('Masukkan API key (skip jika tidak punya):')
    
    if not os.environ.get('GEMINI_API_KEY'):
        k = getpass.getpass('Gemini API Key: ')
        if k: os.environ['GEMINI_API_KEY'] = k
    
    if not os.environ.get('GROQ_API_KEY'):
        k = getpass.getpass('Groq API Key: ')
        if k: os.environ['GROQ_API_KEY'] = k
        
    if not os.environ.get('OPENROUTER_API_KEY'):
        k = getpass.getpass('OpenRouter API Key: ')
        if k: os.environ['OPENROUTER_API_KEY'] = k

# Verify at least one key is set
has_key = any([
    os.environ.get('GEMINI_API_KEY'),
    os.environ.get('GROQ_API_KEY'),
    os.environ.get('OPENROUTER_API_KEY')
])
if has_key:
    print('\n✅ API keys ready!')
else:
    print('\n⚠️ No API keys set. AI analysis will not work.')
    print('   You can still use manual segment input.')

## 📌 Cell 5: Launch Web App 🚀

Jalankan Gradio web app. Setelah cell ini selesai, akan muncul:
- **Link Gradio**: `https://xxxx.gradio.live` — buka di browser HP/laptop
- **Link Local**: `http://localhost:7860`

### ⏱️ Tips:
- Biarkan cell ini tetap running saat proses render
- Colab gratis ada idle timeout ~90 menit — buka tab notebook sesekali biar tetap aktif
- Kalau runtime disconnect, cukup jalankan ulang cell ini (file di Drive tetap aman)

In [ ]:
import sys
repo_dir = '/content/yt-short-clipper-offline'
if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)

os.chdir(repo_dir)

from clipper_core import setup_directories
setup_directories(
    temp_dir='/content/temp',
    output_dir='/content/drive/MyDrive/YTShortClipper/output',
    config_file='/content/drive/MyDrive/YTShortClipper/config.json',
)

from clipper_web import build_ui
demo = build_ui()
demo.launch(share=True, show_error=True)

## 📌 Cell 6 (Opsional): Cek Output

Lihat file yang sudah selesai diproses di Google Drive.

In [ ]:
import os
from pathlib import Path

output_dir = Path('/content/drive/MyDrive/YTShortClipper/output')
if output_dir.exists():
    for date_dir in sorted(output_dir.iterdir(), reverse=True):
        if date_dir.is_dir():
            files = list(date_dir.glob('*.mp4'))
            if files:
                print(f'\n📁 {date_dir.name}/')
                for f in sorted(files, key=lambda x: x.stat().st_mtime, reverse=True):
                    size_mb = f.stat().st_size / (1024*1024)
                    print(f'   🎬 {f.name} ({size_mb:.1f} MB)')
else:
    print('Belum ada output.')